# GPT2 Training with Discord Messages

This notebook demonstrates how to train GPT2 on Discord messages using the Hugging Face Transformers library.

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataLoader, TextDataset, DataCollatorForLanguageModeling
from torch.utils.data import random_split
from pathlib import Path
import json
import logging

from src.logging_config import get_logger

logger = get_logger(__name__)

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load pre-trained GPT2 model and tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Set pad token to EOS token
tokenizer.pad_token = tokenizer.eos_token

model = model.to(device)
print(f"Model loaded: {model_name}")

In [ ]:
# Load and prepare Discord messages
def load_discord_messages(directory):
    """Load all Discord messages from a directory."""
    from src.discord_loader import DiscordLoader
    
    discord_loader = DiscordLoader()
    message_files = discord_loader.discord_find_messages(directory)
    
    all_messages = []
    for file_path in message_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            messages = json.load(f)
            all_messages.extend(messages)
    
    return all_messages

def extract_message_text(messages):
    """Extract text content from Discord messages."""
    texts = []
    for msg in messages:
        content = msg.get('content', '')
        if content:
            texts.append(content)
    return texts

# Load messages
directory = "/Volumes/PortaOne/Datasets/discord_gdpr/Messages/c85338836384628736/messages.json"
messages = load_discord_messages(directory)
texts = extract_message_text(messages)
print(f"Loaded {len(messages)} messages, {len(texts)} with content")

In [ ]:
# Tokenize messages
def tokenize_function(examples):
    return tokenizer(examples, truncation=True, padding='max_length', max_length=512)

tokenized_texts = tokenizer(texts, truncation=True, padding='max_length', max_length=512)
print(f"Tokenized {len(tokenized_texts['input_ids'])} messages")
print(f"Sequence length: {len(tokenized_texts['input_ids'][0])}")

In [ ]:
# Create dataset and split
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    
    def __len__(self):
        return len(self.encodings['input_ids'])
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        return item

dataset = TextDataset(tokenized_texts)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
print(f"Train dataset: {len(train_dataset)}, Val dataset: {len(val_dataset)}")

In [ ]:
# Training setup
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    save_steps=500,
    evaluation_strategy='no',
    save_total_limit=2,
    fp16=not torch.cuda.is_available(),  # Use mixed precision if GPU available
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Training setup complete")

In [ ]:
# Train the model
print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Save the fine-tuned model
model_path = './fine_tuned_gpt2_discord'
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model saved to {model_path}")

In [ ]:
# Generate text samples
def generate_text(prompt, max_length=100, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

# Generate sample text
prompt = "Hello, how are you today?"
generated = generate_text(prompt, max_length=100)
print(f"Prompt: {prompt}")
for i, text in enumerate(generated, 1):
    print(f"\nGenerated {i}: {text}")